# Squggles Data Management.

In this notebook we are gonna be working on data management, of oxford nanopore raw signals, the repository structure is this .

....

to run this notebook if you are running in colab please select a kernel with VRAM as it will use the CUDA system for some tools to improve the speed and performance. 

This notebook conatins 3 big modules, of a series of data preparation for MLL and data annotation for a macro project, that AIMS to develop a model that can predict antibiotic resistance. 

the final output of this notebook is an annotatte CSV file, with the identification of the resistance genes, of interest in the given raw data , in order to train the model.

1. folder structure prep and tools download.
2. Alignment and data annotation.
3. Extraction of the fragments and production of the csv annotation file. 

In [ ]:
# CELL 1: THE STARTING POINT
from google.colab import drive
import os
from pathlib import Path

print("[*] Initiating Zero-State Anchoring to Google Drive...")
drive.mount('/content/drive')

# We create a brand new, dedicated universe for this project.
PROJECT_ROOT = Path('/content/drive/MyDrive/NanoSquiggle_Pipeline').resolve()

# Create the root if it does not exist
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Shift the entire Python execution coordinate system to this root
os.chdir(PROJECT_ROOT)
print(f"[+] Project Origin established at: {PROJECT_ROOT}")

In [ ]:
# CELL 2: DIRECTORY GRAPH CONSTRUCTION
print("📁 Constructing Directory Graph from Scratch...")

directories = [
    "bin/dorado/bin",
    "bin/dorado/lib",
    "data/raw",
    "data/processed/alignments",
    "data/processed/hdf5",
    "data/ref",
    "data/data_index",
    "models",
    "scripts"
]

for d in directories:
    dir_path = PROJECT_ROOT / d
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"    -> Created node: {d}")

print("[+] Topological Tree complete.")

In [ ]:
# CELL 3: ASYMMETRIC BINARY COMPILATION
import urllib.request
import tarfile
import shutil

dorado_target_bin = PROJECT_ROOT / "bin" / "dorado" / "bin" / "dorado"

if not dorado_target_bin.exists():
    print("[*] Dorado binary not found in Vault. Initiating Fast-I/O Assembly...")

    dorado_url = "https://cdn.oxfordnanoportal.com/software/analysis/dorado-0.5.0-linux-x64.tar.gz"

    # Define Ephemeral Paths (Blazing Fast NVMe)
    ephemeral_tar = Path('/content/dorado_temp.tar.gz')
    ephemeral_extract_dir = Path('/content/dorado_temp_extract')
    ephemeral_extract_dir.mkdir(exist_ok=True)

    print("    -> Downloading Dorado to NVMe buffer...")
    urllib.request.urlretrieve(dorado_url, ephemeral_tar)

    print("    -> Extracting archive (High Speed)...")
    with tarfile.open(ephemeral_tar, "r:gz") as tar:
        tar.extractall(path=ephemeral_extract_dir)

    # The extracted folder has a specific versioned name
    extracted_source = ephemeral_extract_dir / "dorado-0.5.0-linux-x64"

    print("    -> Committing binaries to Persistent Vault...")
    # Atomic Moves to Google Drive
    for item in (extracted_source / "bin").iterdir():
        shutil.move(str(item), str(PROJECT_ROOT / "bin" / "dorado" / "bin" / item.name))

    for item in (extracted_source / "lib").iterdir():
        shutil.move(str(item), str(PROJECT_ROOT / "bin" / "dorado" / "lib" / item.name))

    print("    -> Purging Ephemeral Buffer...")
    shutil.rmtree(ephemeral_extract_dir)
    ephemeral_tar.unlink()

    # Enforce Linux executable permissions mathematically
    dorado_target_bin.chmod(0o755)
    print("[+] Dorado mapped successfully to Persistent Storage.")
else:
    print("[SKIP] Dorado binary already exists in Persistent Storage.")

In [ ]:
# CELL 4: THE ASYMMETRIC DATA ORCHESTRATOR
import os
import tarfile
import shutil
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# [IntegrityChecker and NanoSquiggleDownloader remain EXACTLY the same.
# They are mathematically perfect for the sequential download phase.]
class IntegrityChecker:
    @staticmethod
    def validate_magic_bytes(file_path: Path, expected_magic: bytes, offset: int = 0) -> bool:
        try:
            with open(file_path, 'rb') as f:
                f.seek(offset)
                return f.read(len(expected_magic)) == expected_magic
        except Exception as e:
            print(f"[-] Formal Integrity Error: {e}")
            return False

    @staticmethod
    def validate_tar_structure(file_path: Path) -> bool:
        try:
            with tarfile.open(file_path, 'r:*') as tar:
                for _ in tar: break
            return True
        except Exception as e:
            print(f"[-] Logical Integrity Failed: {e}")
            return False

class NanoSquiggleDownloader:
    def __init__(self, output_dir: Path):
        self.output_dir = output_dir

    def download_and_verify(self, url: str, filename: str) -> bool:
        tmp_path = self.output_dir / f"{filename}.tmp"
        file_path = self.output_dir / filename
        print(f"[*] {filename}: Commencing transfer...")
        try:
            with requests.get(url, stream=True, timeout=30) as response:
                response.raise_for_status()
                expected_size = int(response.headers.get('Content-Length', 0))
                with open(tmp_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=65536):
                        f.write(chunk)
        except Exception as e:
            print(f"[-] {filename}: Connection Error: {e}")
            if tmp_path.exists(): tmp_path.unlink()
            return False

        local_size = tmp_path.stat().st_size
        if expected_size > 0 and expected_size != local_size:
            tmp_path.unlink()
            return False

        if not IntegrityChecker.validate_magic_bytes(tmp_path, b'\x1f\x8b'):
            tmp_path.unlink()
            return False

        if not IntegrityChecker.validate_tar_structure(tmp_path):
            tmp_path.unlink()
            return False

        tmp_path.rename(file_path)
        print(f"[+] {filename}: HAC Protocol Passed.")
        return True


class DatasetOrchestrator:
    def __init__(self):
        # In Colab, we establish the root based on the current working directory
        # which we set to the Google Drive folder in the Genesis cell.
        self.project_root = Path(os.getcwd()).resolve()

        self.raw_dir = self.project_root / "data" / "raw"
        self.index_file = self.project_root / "data" / "data_index" / "data_ids.txt"

    def _promote_and_clean(self, target_dir: Path):
        # Note: target_dir here is now the fast NVMe buffer in Colab
        pass_dir = target_dir / "pod5_pass"
        fail_dir = target_dir / "pod5_fail"

        if pass_dir.exists() and pass_dir.is_dir():
            for pod5_file in pass_dir.glob("*.pod5"):
                shutil.move(str(pod5_file), str(target_dir / pod5_file.name))

        for folder in [pass_dir, fail_dir]:
            if folder.exists(): shutil.rmtree(folder)

    def extract_and_organize(self, archive_path: Path, target_dir: Path) -> bool:
        """
        Modified for Google Colab: Asymmetric NVMe Buffering.
        """
        print(f"[*] {target_dir.name}: Commencing Surgical Extraction via NVMe Buffer...")

        # 1. Define the High-Speed Ephemeral Manifold
        ephemeral_buffer = Path(f"/content/extract_buffer_{target_dir.name}")
        ephemeral_buffer.mkdir(exist_ok=True)

        try:
            # 2. Extract to NVMe (Blazing fast random I/O)
            with tarfile.open(archive_path, 'r:*') as tar:
                tar.extractall(path=ephemeral_buffer)

            # 3. Clean and Promote inside NVMe
            self._promote_and_clean(ephemeral_buffer)

            # 4. Migrate State to Persistent Drive (Block transfer)
            print(f"[*] {target_dir.name}: Migrating sanitized data to Persistent Vault...")
            for item in ephemeral_buffer.iterdir():
                shutil.move(str(item), str(target_dir / item.name))

            # 5. Destroy Artifacts
            archive_path.unlink()
            shutil.rmtree(ephemeral_buffer)
            print(f"[+] {target_dir.name}: Asymmetric Extraction complete.")
            return True

        except Exception as e:
            print(f"[-] {target_dir.name}: Extraction/Hygiene Failed: {e}")
            if ephemeral_buffer.exists(): shutil.rmtree(ephemeral_buffer)
            return False

    def process_strain(self, strain_id: str, url: str) -> bool:
        strain_dir = self.raw_dir / strain_id
        strain_dir.mkdir(parents=True, exist_ok=True)

        sentinel_file = strain_dir / ".download_complete"
        filename = f"{strain_id}.tar.gz"
        archive_path = strain_dir / filename

        if sentinel_file.exists():
            print(f"[SKIP] {strain_id}: Sentinel found. Dataset is verified and clean.")
            return True

        downloader = NanoSquiggleDownloader(output_dir=strain_dir)
        if not downloader.download_and_verify(url, filename):
            return False

        if not self.extract_and_organize(archive_path, strain_dir):
            return False

        sentinel_file.touch()
        print(f"[SUCCESS] {strain_id}: Lifecycle complete.")
        return True

    def execute_pipeline(self, max_workers: int = 2):
        # NOTE: Reduced max_workers to 2. Google Drive API penalizes high concurrency.
        if not self.index_file.exists():
            print(f"[-] FATAL: Index file not found at {self.index_file}")
            return

        with open(self.index_file, 'r') as f:
            strain_ids = [line.strip() for line in f if line.strip()]

        base_url = "https://data.narodni-repozitar.cz/general/datasets/dj8ys-a4r49/files/"
        tasks = {sid: f"{base_url}{sid}_pod5.tar.gz" for sid in strain_ids}

        print(f"[*] Booting Unified Data Engine. Target strains: {len(tasks)}")

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self.process_strain, sid, url): sid for sid, url in tasks.items()}
            for future in as_completed(futures):
                strain_id = futures[future]
                try:
                    future.result()
                except Exception as e:
                    print(f"[-] Unhandled Thread Exception on {strain_id}: {e}")

if __name__ == "__main__":
    engine = DatasetOrchestrator()
    # Execute with 2 workers to respect Drive API constraints
    engine.execute_pipeline(max_workers=2)

In [ ]:
# CELL 1: INJECT SYSTEM DEPENDENCIES
# We must install samtools into the Colab Ubuntu instance.
!apt-get update
!apt-get install -y samtools
!samtools --version

In [ ]:
# CELL 2: THE BASECALLING ORCHESTRATOR
import subprocess
import os
import sys
import re
import threading
from pathlib import Path

class DoradoRunner:
    """
    Process Manager for Executing dorado basecalling, avoiding Deadlock.
    Adapted for Google Colab Execution Context.
    """

    def __init__(self):
        self.project_root = Path.cwd().resolve()

        self.raw_dir = self.project_root / "data" / "raw"
        self.processed_dir = self.project_root / "data" / "processed" / "alignments"

        self.dorado_bin = self.project_root / "bin" / "dorado" / "bin" / "dorado"
        self.reference_fasta = self.project_root / "data" / "ref" / "resistance_genes.fasta"

        self.processed_dir.mkdir(parents=True, exist_ok=True)

        # --- THE JIT STATE ASSERTION ---
        # We mathematically force the host kernel to grant execution rights.

        try:
            self.dorado_bin.chmod(0o755)
            print("[+] Binary execution vector asserted (chmod 755).")
        except Exception as e:
            print(f"[-] Warning: Could not assert execution vector: {e}")

        print(f"[*] Geometry Engine initialized at: {self.project_root}")


    def _get_env(self) -> dict:
        env = os.environ.copy()
        local_lib = str(self.project_root / "bin" / "dorado" / "lib")
        current_ld = env.get("LD_LIBRARY_PATH", "")
        env["LD_LIBRARY_PATH"] = f"{local_lib}:{current_ld}" if current_ld else local_lib
        return env

    def _monitor_telemetry(self, pipe, strain_id: str):
        # Expanded pattern to handle different Dorado versions and units
        throughput_pattern = re.compile(r'([0-9.]+\s+[kM]?(?:samples|bp)/s)')

        for line in iter(pipe.readline, b''):
            decoded_line = line.decode('utf-8', errors='ignore').strip()

            match = throughput_pattern.search(decoded_line)
            if match:
                throughput = match.group(1)
                # update the progress in-place
                sys.stderr.write(f"\r[*] {strain_id} Speed: {throughput}       ")
                sys.stderr.flush()
            elif any(x in decoded_line.lower() for x in ["error", "warning", "fatal", "info"]):
                if "samples/s" not in decoded_line and "bp/s" not in decoded_line:
                    sys.stderr.write(f"\n[*] {strain_id} Status: {decoded_line}\n")

        sys.stderr.write("\n")

    def execute_pipeline(self, strain_id: str, pod5_dir: Path) -> bool:
        tmp_bam = self.processed_dir / f"{strain_id}.tmp.bam"
        final_bam = self.processed_dir / f"{strain_id}.bam"
        samtools_log = self.processed_dir / f"{strain_id}_samtools.log" # NEW: Physical Log Manifold

        if final_bam.exists():
            print(f"[SKIP] {strain_id}: Alignment artifact present.")
            return True

        dorado_cmd = [
            str(self.dorado_bin), "basecaller",
            "hac",
            str(pod5_dir),
            "--reference", str(self.reference_fasta),
            "--emit-moves",
            "--emit-sam"
        ]

        samtools_cmd = ["samtools", "sort", "-m", "2G", "-o", str(tmp_bam), "-"]

        print(f"[*] {strain_id}: Booting Compute Manifold...")

        try:
            p_dorado = subprocess.Popen(
                dorado_cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                env=self._get_env()
            )

            # THE FIX: Open a physical file to generate a valid OS fileno
            with open(samtools_log, "w") as s_log:
                p_samtools = subprocess.Popen(
                    samtools_cmd,
                    stdin=p_dorado.stdout,
                    stderr=s_log  # Linux Kernel routes bytes directly to the SSD
                )

                p_dorado.stdout.close()

                monitor_thread = threading.Thread(
                    target=self._monitor_telemetry,
                    args=(p_dorado.stderr, strain_id),
                    daemon=True
                )
                monitor_thread.start()

                samtools_exit = p_samtools.wait()
                dorado_exit = p_dorado.wait()
                monitor_thread.join()

            # Post-Execution Atomic Checks
            if dorado_exit == 0 and samtools_exit == 0:
                tmp_bam.rename(final_bam)
                print(f"[+] {strain_id}: Atomic Commit successful. Generating index...")

                # Cleanup the log if execution was perfect to save space
                if samtools_log.exists(): samtools_log.unlink()

                try:
                    subprocess.run(["samtools", "index", str(final_bam)], check=True)
                    return True
                except subprocess.CalledProcessError as e:
                    print(f"[-] {strain_id}: Indexing Failed: {e}. Purging unsorted BAM.")
                    if final_bam.exists(): final_bam.unlink()
                    return False
            else:
                print(f"[-] {strain_id}: Pipeline Failed (Dorado: {dorado_exit}, Samtools:{samtools_exit})")
                print(f"    -> Check {samtools_log.name} for detailed kernel errors.")
                if tmp_bam.exists(): tmp_bam.unlink()
                return False

        except Exception as e:
            print(f"[-] FATAL IPC ERROR: {e}")
            if tmp_bam.exists(): tmp_bam.unlink()
            return False

    def run(self):
        if not self.raw_dir.exists():
            print(f"[-] No raw manifold found at {self.raw_dir}")
            return

        for strain_folder in self.raw_dir.iterdir():
            if not strain_folder.is_dir():
                continue

            sentinel = strain_folder / ".download_complete"
            if not sentinel.exists():
                print(f"[!] Skipping {strain_folder.name}: Sentinel missing.")
                continue

            self.execute_pipeline(strain_folder.name, strain_folder)

if __name__ == "__main__":
    DoradoRunner().run()